## WonderBox 포함 분석 Ver.3 : 년평균 평점-매출 데이터 --> WonderBox 컬렉션 합치기 --> Contribution --> 증감패턴  

## Collection1 기준으로 집계

In [218]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [220]:
df = pd.read_csv('review-vc_sales-by_collection_category_20250429.csv')

In [222]:
df['coalesce_collection'].nunique()

641

In [224]:
filtered_df = df[['yr', 'financial_category', 'coalesce_collection', 'written_avg_rating', 'sales_amount']]
filtered_df = filtered_df.rename(columns={'financial_category':'category'})
filtered_df = filtered_df.rename(columns={'coalesce_collection':'collection1'})
filtered_df = filtered_df.rename(columns={'written_avg_rating':'rating'})
filtered_df = filtered_df.rename(columns={'sales_amount':'sales'})

In [226]:
filtered_df

,yr,category,collection1,rating,sales
0,2022,Box Springs,Edgar 4in,4.568966,976224.39
1,2022,Box Springs,Daniel 9in,4.216216,533974.46
2,2022,Box Springs,Adrianne,NaN,913.93
3,2022,Box Springs,Jayanna BIFD 7.5in,4.725000,1114260.42
4,2022,Box Springs,NT BIFD 9in,NaN,NaN
...,...,...,...,...,...
2519,2025,Toppers,2in Swirl Copper,5.000000,4590.32
2520,2025,Toppers,1.5in Rejuvenator,NaN,NaN
2521,2025,Toppers,1in GT Zoned Gel,NaN,NaN
2522,2025,Toppers,1.5in Copper Convoluted,NaN,NaN


In [231]:
agg_df = filtered_df.groupby(['yr','category','collection1']).agg({
    'rating': 'mean',
    'sales': 'sum'
}).reset_index()

In [233]:
agg_df['collection1'].nunique()

641

In [235]:
agg_df[agg_df['collection1']=='4in MyGel']

,yr,category,collection1,rating,sales
560,2022,Toppers,4in MyGel,4.075472,436691.26
1155,2023,Toppers,4in MyGel,3.482759,292672.49
1791,2024,Toppers,4in MyGel,3.541667,203213.91
2433,2025,Toppers,4in MyGel,3.850000,33509.66


In [237]:
# yr 값을 컬럼으로 변환 (pivot)
pivot_df = agg_df.pivot_table(index=['category','collection1'],
                              columns='yr',
                              values=['rating','sales'])

In [239]:
# 컬럼 이름을 정리 (multi-index 해제)
#pivot_df.columns = [f"{val}_{yr}" for val, yr in pivot_df.columns]
pivot_df.columns = [f"{yr}_{val}" for val, yr in pivot_df.columns]

In [241]:
pivot_df

2022_rating  2023_rating  2024_rating  \
category    collection1                                                  
Box Springs 5in OOT Box Spring           NaN          NaN     2.950000   
            9in OOT Box Spring           NaN          NaN     3.242424   
            Adrianne                     NaN          NaN          NaN   
            Annemarie               4.186170     4.319767     4.341121   
            Armita 5in              4.099029     3.837037     3.346939   
...                                      ...          ...          ...   
Toppers     4in MyGel               4.075472     3.482759     3.541667   
            4in PRMF                     NaN          NaN          NaN   
            4in PRMF w Cover        5.000000          NaN          NaN   
            4in Swirl Gel           3.500000     3.666667     3.800000   
            4in TorsoTec Copper          NaN          NaN          NaN   

                                 2025_rating   2022_sales  2023_sales  \
category    collection1                                                 
Box Springs 5in OOT Box Spring      3.205882          NaN         NaN   
            9in OOT Box Spring      3.896552          NaN         NaN   
            Adrianne                     NaN       913.93        0.00   
            Annemarie               4.166667   2839354.78  2311639.79   
            Armita 5in              3.666667  11109930.21  5701328.78   
...                                      ...          ...         ...   
Toppers     4in MyGel               3.850000    436691.26   292672.49   
            4in PRMF                     NaN     14280.09    15607.23   
            4in PRMF w Cover             NaN         0.00        0.00   
            4in Swirl Gel           1.333333    269305.47   153721.07   
            4in TorsoTec Copper     3.000000         0.00        0.00   

                                 2024_sales  2025_sales  
category    collection1                                  
Box Springs 5in OOT Box Spring      5258.64   241483.44  
            9in OOT Box Spring      7379.26   286722.15  
            Adrianne                   0.00        0.00  
            Annemarie            2599136.15   465597.56  
            Armita 5in            410591.11    10137.96  
...                                     ...         ...  
Toppers     4in MyGel             203213.91    33509.66  
            4in PRMF                 120.99        0.00  
            4in PRMF w Cover           0.00        0.00  
            4in Swirl Gel         174365.99    33740.54  
            4in TorsoTec Copper        0.00        0.00  

[642 rows x 8 columns]

In [243]:
pivot_df.reset_index()
pivot_df.to_csv('review_sales_wonderbox_out_02.csv')

## 증감 패턴 구하기

In [391]:
df = pd.read_csv('review_sales_wonderbox_out_02.csv')

In [393]:
df['2022_weight'] = df['2022_rating'] * df['2022_sales']
df['2023_weight'] = df['2023_rating'] * df['2023_sales']
df['2024_weight'] = df['2024_rating'] * df['2024_sales']
df['2025_weight'] = df['2025_rating'] * df['2025_sales']

In [395]:
y2022_total_weight = df['2022_weight'].sum()
df['2022_contribution'] = df['2022_weight']/y2022_total_weight

y2023_total_weight = df['2023_weight'].sum()
df['2023_contribution'] = df['2023_weight']/y2023_total_weight

y2024_total_weight = df['2024_weight'].sum()
df['2024_contribution'] = df['2024_weight']/y2024_total_weight

y2025_total_weight = df['2025_weight'].sum()
df['2025_contribution'] = df['2025_weight']/y2025_total_weight

In [397]:
df

,category,collection1,2022_rating,2023_rating,2024_rating,2025_rating,2022_sales,2023_sales,2024_sales,2025_sales,2022_weight,2023_weight,2024_weight,2025_weight,2022_contribution,2023_contribution,2024_contribution,2025_contribution
0,Box Springs,5in OOT Box Spring,NaN,NaN,2.950000,3.205882,NaN,NaN,5258.64,241483.44,NaN,NaN,1.551299e+04,7.741675e+05,NaN,NaN,0.000012,0.002222
1,Box Springs,9in OOT Box Spring,NaN,NaN,3.242424,3.896552,NaN,NaN,7379.26,286722.15,NaN,NaN,2.392669e+04,1.117228e+06,NaN,NaN,0.000019,0.003207
2,Box Springs,Adrianne,NaN,NaN,NaN,NaN,913.93,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Box Springs,Annemarie,4.186170,4.319767,4.341121,4.166667,2839354.78,2311639.79,2599136.15,465597.56,1.188602e+07,9.985746e+06,1.128317e+07,1.939990e+06,0.004663,0.004951,0.009031,0.005569
4,Box Springs,Armita 5in,4.099029,3.837037,3.346939,3.666667,11109930.21,5701328.78,410591.11,10137.96,4.553993e+07,2.187621e+07,1.374223e+06,3.717252e+04,0.017865,0.010846,0.001100,0.000107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637,Toppers,4in MyGel,4.075472,3.482759,3.541667,3.850000,436691.26,292672.49,203213.91,33509.66,1.779723e+06,1.019308e+06,7.197159e+05,1.290122e+05,0.000698,0.000505,0.000576,0.000370
638,Toppers,4in PRMF,NaN,NaN,NaN,NaN,14280.09,15607.23,120.99,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
639,Toppers,4in PRMF w Cover,5.000000,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.000000e+00,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
640,Toppers,4in Swirl Gel,3.500000,3.666667,3.800000,1.333333,269305.47,153721.07,174365.99,33740.54,9.425691e+05,5.636439e+05,6.625908e+05,4.498739e+04,0.000370,0.000279,0.000530,0.000129


## Rating, Contribution 증감 패턴 구하기

In [399]:
# 1. rating_slope 컬럼 추가: 결측치 제외 후 연간 rating으로 기울기 계산
def calc_rating_slope(row):
    ratings = [row['2022_rating'], row['2023_rating'], row['2024_rating'], row['2025_rating']]
    years = [2022, 2023, 2024, 2025]

    # 결측치 아닌 값만 골라내기
    valid_x = []
    valid_y = []

    for i, r in enumerate(ratings):
        if pd.notna(r): 
            valid_x.append(i)
            valid_y.append(r)

    if len(valid_x) <= 1:
        return 0

    x = np.array(valid_x).reshape(-1,1)
    y = np.array(valid_y)

    model = LinearRegression()
    model.fit(x,y)
    return model.coef_[0]

In [401]:
df['rating_slope'] = df.apply(calc_rating_slope, axis=1)

In [403]:
def determine_trend(slope):
    if slope > 0: 
        return 1
    elif slope < 0:
        return -1
    else: 
        return 0

In [405]:
df['rating_trend'] = df['rating_slope'].apply(determine_trend)

In [411]:
df[df['collection1'].str.contains('Armita')]

,category,collection1,2022_rating,2023_rating,2024_rating,2025_rating,2022_sales,2023_sales,2024_sales,2025_sales,2022_weight,2023_weight,2024_weight,2025_weight,2022_contribution,2023_contribution,2024_contribution,2025_contribution,rating_slope,rating_trend
4,Box Springs,Armita 5in,4.099029,3.837037,3.346939,3.666667,11109930.21,5701328.78,410591.11,10137.96,4.553993e+07,2.187621e+07,1.374223e+06,3.717252e+04,0.017865,0.010846,0.001100,0.000107,-0.178719,-1
5,Box Springs,Armita 7in,3.707483,3.604396,3.400000,3.250000,6203976.18,4090373.05,398262.53,52703.33,2.300114e+07,1.474332e+07,1.354093e+06,1.712858e+05,0.009023,0.007310,0.001084,0.000492,-0.157684,-1
6,Box Springs,Armita 9in,3.977918,3.980940,3.543478,2.794118,27701764.81,16637443.49,2800984.63,268419.37,1.101953e+08,6.623267e+07,9.925228e+06,7.499953e+05,0.043229,0.032838,0.007944,0.002153,-0.398886,-1
7,Box Springs,Armita QA 5in,4.444444,3.875000,3.709091,3.111111,404525.37,2573436.41,1837738.86,238965.26,1.797891e+06,9.972066e+06,6.816340e+06,7.434475e+05,0.000705,0.004944,0.005456,0.002134,-0.416591,-1
8,Box Springs,Armita QA 7in,4.300000,3.813008,3.272727,3.600000,243985.41,2057636.78,910367.63,160570.51,1.049137e+06,7.845786e+06,2.979385e+06,5.780538e+05,0.000412,0.003890,0.002385,0.001659,-0.264028,-1
9,Box Springs,Armita QA 9in,3.898551,3.912371,4.050562,3.970588,2733777.47,7475034.64,6697325.62,846741.70,1.065777e+07,2.924511e+07,2.712793e+07,3.362063e+06,0.004181,0.014500,0.021714,0.009651,0.035430,1


In [375]:
# 2. contribution_slope 컬럼 추가: 결측치 제외 후 연간 contribution 기울기 계산
def calc_contribution_slope(row):
    contributions = [row['2022_contribution'], row['2023_contribution'], row['2024_contribution'], row['2025_contribution']]
    years = [2022, 2023, 2024, 2025]

    # 결측치 아닌 값만 골라내기
    valid_x = []
    valid_y = []

    for i, r in enumerate(contributions):
        if pd.notna(r): 
            valid_x.append(i)
            valid_y.append(r)

    if len(valid_x) <= 1:
        return 0

    x = np.array(valid_x).reshape(-1,1)
    y = np.array(valid_y)

    model = LinearRegression()
    model.fit(x,y)
    return model.coef_[0]


In [383]:
df['contribution_slope'] = df.apply(calc_contribution_slope, axis=1)

In [385]:
df['contribution_trend'] = df['contribution_slope'].apply(determine_trend)

In [387]:
df

,category,collection1,2022_rating,2023_rating,2024_rating,2025_rating,2022_sales,2023_sales,2024_sales,2025_sales,...,2024_weight,2025_weight,2022_contribution,2023_contribution,2024_contribution,2025_contribution,rating_slope,rating_trend,contribution_slope,contribution_trend
0,Box Springs,5in OOT Box Spring,NaN,NaN,2.950000,3.205882,NaN,NaN,5258.64,241483.44,...,1.551299e+04,7.741675e+05,NaN,NaN,0.000012,0.002222,0.255882,1,0.002210,1
1,Box Springs,9in OOT Box Spring,NaN,NaN,3.242424,3.896552,NaN,NaN,7379.26,286722.15,...,2.392669e+04,1.117228e+06,NaN,NaN,0.000019,0.003207,0.654127,1,0.003188,1
2,Box Springs,Adrianne,NaN,NaN,NaN,NaN,913.93,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0,0.000000,0
3,Box Springs,Annemarie,4.186170,4.319767,4.341121,4.166667,2839354.78,2311639.79,2599136.15,465597.56,...,1.128317e+07,1.939990e+06,0.004663,0.004951,0.009031,0.005569,-0.003716,-1,0.000680,1
4,Box Springs,Armita 5in,4.099029,3.837037,3.346939,3.666667,11109930.21,5701328.78,410591.11,10137.96,...,1.374223e+06,3.717252e+04,0.017865,0.010846,0.001100,0.000107,-0.178719,-1,-0.006302,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637,Toppers,4in MyGel,4.075472,3.482759,3.541667,3.850000,436691.26,292672.49,203213.91,33509.66,...,7.197159e+05,1.290122e+05,0.000698,0.000505,0.000576,0.000370,-0.061751,-1,-0.000091,-1
638,Toppers,4in PRMF,NaN,NaN,NaN,NaN,14280.09,15607.23,120.99,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0,0.000000,0
639,Toppers,4in PRMF w Cover,5.000000,NaN,NaN,NaN,0.00,0.00,0.00,0.00,...,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,0,0.000000,0
640,Toppers,4in Swirl Gel,3.500000,3.666667,3.800000,1.333333,269305.47,153721.07,174365.99,33740.54,...,6.625908e+05,4.498739e+04,0.000370,0.000279,0.000530,0.000129,-0.636667,-1,-0.000047,-1


In [389]:
df.to_csv('review_sales_wonderbox_pattern_out_01.csv')